In [ ]:
!pip install faster-whisper
!apt-get install ffmpeg -y
!pip install pyngrok
!pip install dotenv
!pip install rapidfuzz


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from faster_whisper import WhisperModel

In [ ]:

# Set your ngrok authtoken
from pyngrok import ngrok
from dotenv import load_dotenv

load_dotenv()
ngrok.set_auth_token(GNORK_AUTH_TOKEN)


In [13]:
model = WhisperModel("medium", compute_type="int8")  # Use "cpu" if needed

In [ ]:
from rapidfuzz import fuzz, process

# List of names you want the model to recognize
KNOWN_NAMES = ["Ankit", "Aman", "Asmit", "Dhaubanjar", "Pokhrel", "Phuyal", "SIC", "Pulchowk", ]

def correct_transcription(transcript: str, threshold: int = 60) -> str:
    words = transcript.split()
    corrected_words = []

    for word in words:
        match, score, _ = process.extractOne(word, KNOWN_NAMES, scorer=fuzz.ratio)
        if score >= threshold:
            corrected_words.append(match)
        else:
            corrected_words.append(word)

    return " ".join(corrected_words)


In [ ]:
# Create Flask app
app = Flask(__name__)

@app.route("/transcribe", methods=["POST"])
def transcribe_audio():
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400

    file = request.files['file']
    file_path = "/content/temp.wav"
    file.save(file_path)

    segments, _ = model.transcribe(file_path, language="ne", task="translate", vad_filter = True)
    result = " ".join([segment.text for segment in segments])


    return jsonify({"translation": result})




In [15]:
# Open ngrok tunnel
public_url = ngrok.connect(5000)
print(f"🔥 Your public URL: {public_url}")

# Run the app
app.run(port=5000)

🔥 Your public URL: NgrokTunnel: "https://f46c-34-173-65-36.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/May/2025 07:05:05] "POST /transcribe HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2025 07:06:08] "POST /transcribe HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2025 07:06:37] "POST /transcribe HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2025 07:07:05] "POST /transcribe HTTP/1.1" 200 -
